# Stage 2: Deprivation Score Silver Layer

This notebook creates the cleaned Silver layer for the Index of Deprivation enrichment dataset.

The dataset is used to provide socioeconomic context for the selected police forces:
- West Midlands
- Thames Valley
- Surrey
- Cumbria

The Silver layer standardises column names, cleans numeric deprivation ranking fields, maps local authorities to police forces, validates the data, and writes the cleaned table to Snowflake.

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.connector.pandas_tools import write_pandas

session = get_active_session()

STAGE = "@CRIME_PIPELINE.RAW.HOUSE_PRICE_STAGE"

## 1. Read IMD Dataset from Snowflake Stage

The cleaned IMD CSV file is read from the Snowflake internal stage.

In [ ]:
imd = pd.read_csv(
    session.file.get_stream(
        f"{STAGE}/imd_lad_clean.csv"
    ),
    dtype=str,
    low_memory=False
)

print("Rows loaded:", len(imd))
imd.head()

## 2. Standardise Column Names

Column names are converted to uppercase and stripped of whitespace to ensure consistency across the pipeline.

In [ ]:
imd.columns = imd.columns.str.upper().str.strip()

print(imd.columns.tolist())

## 3. Clean Text and Numeric Fields

Local authority fields are cleaned, and deprivation ranking columns are converted into numeric format.

In [ ]:
imd["LAD_CODE"] = imd["LAD_CODE"].astype(str).str.strip()
imd["LAD_NAME"] = imd["LAD_NAME"].astype(str).str.strip()

rank_cols = [
    "IMD_RANK",
    "INCOME_RANK",
    "EMPLOYMENT_RANK",
    "EDUCATION_RANK",
    "HEALTH_RANK",
    "CRIME_RANK",
    "HOUSING_BARRIERS_RANK",
    "LIVING_ENVIRONMENT_RANK",
    "IDACI_RANK",
    "IDAOPI_RANK"
]

for col in rank_cols:
    if col in imd.columns:
        imd[col] = pd.to_numeric(imd[col], errors="coerce")

## 4. Map Local Authorities to Police Forces

Each local authority is mapped to one of the selected police forces. 

In [ ]:
def map_police_force(authority):

    west_midlands = [
        "Birmingham", "Coventry", "Dudley",
        "Sandwell", "Solihull",
        "Walsall", "Wolverhampton"
    ]

    surrey = [
        "Elmbridge", "Epsom and Ewell", "Guildford",
        "Mole Valley", "Reigate and Banstead",
        "Runnymede", "Spelthorne",
        "Surrey Heath", "Tandridge",
        "Waverley", "Woking"
    ]

    thames_valley = [
        "Bracknell Forest", "Reading", "Slough",
        "West Berkshire", "Windsor and Maidenhead",
        "Wokingham", "Milton Keynes",
        "Cherwell", "Oxford",
        "South Oxfordshire",
        "Vale of White Horse",
        "West Oxfordshire"
    ]

    cumbria = [
        "Cumberland",
        "Westmorland and Furness"
    ]

    if authority in west_midlands:
        return "West Midlands"
    elif authority in surrey:
        return "Surrey"
    elif authority in thames_valley:
        return "Thames Valley"
    elif authority in cumbria:
        return "Cumbria"
    else:
        return None


imd["POLICE_FORCE"] = imd["LAD_NAME"].apply(map_police_force)

## 5. Create Silver IMD Dataset

Only records belonging to the selected police forces are retained.

In [ ]:
silver_imd = imd[
    imd["POLICE_FORCE"].notna()
].copy()

print("Silver rows:", len(silver_imd))
silver_imd.head()

## 6. Validation Checks

The Silver layer is validated using row counts, null checks, duplicate checks, and police force distribution checks.

In [ ]:
print("Null checks:")
print(silver_imd.isnull().sum())

In [ ]:
duplicates = silver_imd.duplicated(
    subset=["LAD_CODE"]
).sum()

print("Duplicate LAD_CODE rows:", duplicates)

In [ ]:
print(
    silver_imd["POLICE_FORCE"]
    .value_counts()
)

## 7. Write Silver Table to Snowflake

The cleaned IMD Silver dataset is written to the Snowflake CLEAN schema.

In [ ]:
conn = session.connection

success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=silver_imd,
    table_name="SILVER_IMD_LAD",
    database="CRIME_PIPELINE",
    schema="CLEAN",
    auto_create_table=True,
    overwrite=True
)

print("Silver IMD table written successfully")
print("Rows:", nrows)
print("Chunks:", nchunks)